# 07_cicd_pipeline.ipynb – CI/CD Pipeline and Monitoring Execution

This notebook completes the project by simulating a Continuous Integration and Continuous Deployment (CI/CD) process. The goal is to show how model monitoring can be tied into a pipeline that helps decide whether a model is ready for approval or needs further review.

## What This Notebook Does

1. Creates a monitoring schedule for the SageMaker endpoint that was deployed earlier.
2. Checks if the monitoring job has run and examines the result.
3. Based on the result, either approves the model or flags it.
4. (Optional) Deletes the endpoint after monitoring is complete to avoid extra charges.

## Why This Is Important

Even if a model performs well during training, real-world data can change. If the model receives unexpected inputs after deployment, it could behave incorrectly. That’s why monitoring live model inputs and outputs is a key part of any reliable ML system.

This notebook:

- Automatically checks the quality of the data going into the deployed model
- Uses monitoring results to decide whether the model should be approved
- Simulates part of a real-world ML deployment workflow

The model was trained and saved in `03_model_training.ipynb`.  
The endpoint was deployed and set up to capture data in `05_monitoring_and_registry.ipynb`.  
This notebook (`07_cicd_pipeline.ipynb`) builds on those steps by running monitoring and using its results as part of the deployment decision process.


## Sagemaker Setup

In [8]:
import boto3
from sagemaker import get_execution_role
from sagemaker.session import Session
from sagemaker.model_monitor import DefaultModelMonitor

# Session and role setup
region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)
sagemaker_session = Session(boto_session=boto_session)
sagemaker_client = boto_session.client("sagemaker")
role = get_execution_role()

# S3 bucket and endpoint
bucket = sagemaker_session.default_bucket()
endpoint_name = "readmission-endpoint11"


## Deploy Model

In [9]:
from sagemaker.model_monitor import DataCaptureConfig
from sagemaker.sklearn.model import SKLearnModel

model_artifact_uri = f"s3://{bucket}/diabetes/registry/model.tar.gz"

sklearn_model = SKLearnModel(
    model_data=model_artifact_uri,
    role=role,
    entry_point="inference.py",
    framework_version="1.0-1",
    sagemaker_session=sagemaker_session
)

data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=f"s3://{bucket}/monitoring/data_capture",
    capture_options=["Input", "Output"]
)

predictor = sklearn_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=endpoint_name,
    data_capture_config=data_capture_config
)

print("Model deployed to endpoint:", endpoint_name)


-------!Model deployed to endpoint: readmission-endpoint11


## Test Deployment

In [13]:
import pandas as pd
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

X_val = pd.read_pickle("data/X_val.pkl")

# Select a single row and ensure integer types
row = X_val.iloc[0].astype(int)

# Configure predictor for CSV input and output
predictor.serializer = CSVSerializer()
predictor.deserializer = CSVDeserializer()

# Make prediction
response = predictor.predict(row.tolist())

print("Response:", response)


Response: [['0']]


## Monitoring in CI/CD

In [27]:
from sagemaker.model_monitor import DefaultModelMonitor, CronExpressionGenerator
from sagemaker.model_monitor.dataset_format import DatasetFormat
from botocore.exceptions import ClientError
import boto3

monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sagemaker_session
)

In [28]:
# Hourly

from sagemaker.model_monitor import EndpointInput, CronExpressionGenerator

baseline_s3_uri = f"s3://{bucket}/monitoring/baseline"

endpoint_input = EndpointInput(
    endpoint_name=endpoint_name,
    destination="/opt/ml/processing/input"
)

monitor.create_monitoring_schedule(
    monitor_schedule_name="readmission-monitor-schedule",
    endpoint_input=endpoint_input,
    output_s3_uri=f"s3://{bucket}/monitoring/reports",
    statistics=baseline_s3_uri + "/statistics.json",
    constraints=baseline_s3_uri + "/constraints.json",
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True
)

print("Monitoring schedule created successfully.")


Monitoring schedule created successfully.


In [31]:
import boto3
from botocore.exceptions import ClientError
import time # For polling if needed

def check_monitor_status(schedule_name):
    # Checks and prints the status of the latest monitoring job execution
    try:
        response = sagemaker_client.describe_monitoring_schedule(MonitoringScheduleName=schedule_name)
        status = response.get("LastMonitoringExecutionSummary", {}).get("MonitoringExecutionStatus", "Unknown")
        print(f"Monitor Status: {status}")
        return status
    except ClientError as e:
        print(f"Error checking monitor status for {schedule_name}: {e}")
        return "ERROR"

# Call the function to check status
monitor_schedule_name = 'readmission-monitor-schedule'
monitor_status = check_monitor_status(monitor_schedule_name)

# You can add a simple polling loop if you want to wait for completion
# print("Waiting for monitor job to complete...")
# while monitor_status not in ["Completed", "CompletedWithViolations", "Failed", "ERROR"]:
#     time.sleep(60) # Wait 60 seconds before re-checking
#     monitor_status = check_monitor_status(monitor_schedule_name)
# print(f"Monitor job finished with status: {monitor_status}")

Monitor Status: Failed


## Run Monitor Every Minute (for Demo)

In [26]:
# from sagemaker.model_monitor import EndpointInput
# from botocore.exceptions import ClientError
# import time 

# # # Delete existing schedule if it has the same name 
# # monitor_schedule_name = "readmission-monitor-schedule"
# # try:
# #     sagemaker_client.delete_monitoring_schedule(MonitoringScheduleName=monitor_schedule_name)
# #     print(f"Existing monitoring schedule '{monitor_schedule_name}' deleted. Waiting for deletion to complete...")
# #     # Give AWS a moment to process the deletion
# #     time.sleep(10)
# # except ClientError as e:
# #     if "ValidationException" in str(e) and "Could not find monitoring schedule" in str(e):
# #         print(f"Monitoring schedule '{monitor_schedule_name}' does not exist, proceeding to create.")
# #     else:
# #         print(f"Error deleting monitoring schedule '{monitor_schedule_name}': {e}")
# # except Exception as e:
# #     print(f"An unexpected error occurred during schedule deletion: {e}")

# monitor_schedule_name = "readmission-monitor-every-minute"

# # Define the cron expression for "every minute" using the most common AWS format.
# # This means: every minute, every hour, no specific day of month (?), every month, every day of week, every year.
# every_minute_cron_expression = "cron(* * ? * * *)"


# # Baseline S3 URI (ensure this path is correct from 05_monitoring_and_registry.ipynb)
# baseline_s3_uri = f"s3://{bucket}/monitoring/baseline"

# endpoint_input = EndpointInput(
#     endpoint_name=endpoint_name,
#     destination="/opt/ml/processing/input" # This is typical for model monitor inputs
# )

# print(f"Creating monitoring schedule to run every minute with cron: {every_minute_cron_expression}...")

# monitor.create_monitoring_schedule(
#     monitor_schedule_name=monitor_schedule_name,
#     endpoint_input=endpoint_input,
#     output_s3_uri=f"s3://{bucket}/monitoring/reports", # Where reports are stored
#     statistics=baseline_s3_uri + "/statistics.json",
#     constraints=baseline_s3_uri + "/constraints.json",
#     # Pass the precisely defined cron expression
#     schedule_cron_expression=every_minute_cron_expression,
#     enable_cloudwatch_metrics=True
# )

# print(f"Monitoring schedule '{monitor_schedule_name}' created successfully to run every minute.")


## Monitoring Execution Status

In [16]:
# # Define function

# def check_monitoring_execution_status(schedule_name):
#     response = sagemaker_client.describe_monitoring_schedule(MonitoringScheduleName=schedule_name)
#     exec_summary = response.get("LastMonitoringExecutionSummary", {})
#     status = exec_summary.get("MonitoringExecutionStatus", "Unknown")
#     print("Monitoring status:", status)
#     return status


In [18]:
# # Call function

# monitor_schedule_name="readmission-monitor-schedule"

# try:
#     status = check_monitoring_execution_status(monitor_schedule_name)
#     if status == "Completed":
#         print("CI/CD Success: Monitoring passed. Proceeding with deployment.")
#     elif status == "CompletedWithViolations":
#         print("CI/CD Warning: Violations detected. Needs review.")
#     else:
#         raise RuntimeError(f"CI/CD Failure: Monitoring ended with status: {status}")
# except Exception as e:
#     print("CI/CD failed:", e)


Monitoring status: Failed
CI/CD failed: CI/CD Failure: Monitoring ended with status: Failed


## Parse Monitor Outputs

In [ ]:
# import json
# import boto3

# s3 = boto3.client("s3")
# output_prefix = "monitoring/output"
# monitor_output_keys = s3.list_objects_v2(Bucket=bucket, Prefix=output_prefix)

# # Find the most recent statistics.json
# stats_key = None
# for obj in monitor_output_keys.get("Contents", []):
#     if obj["Key"].endswith("statistics.json"):
#         stats_key = obj["Key"]

# if stats_key:
#     s3.download_file(bucket, stats_key, "statistics.json")
#     with open("statistics.json", "r") as f:
#         stats = json.load(f)
#     record_count = stats.get("dataset", {}).get("item_count")
#     print(f"Number of records analyzed by monitor: {record_count}")
# else:
#     print("No statistics.json found yet.")


## Alert for Low Record Count

In [ ]:
# if record_count and record_count < 10:
#     raise RuntimeError("Monitoring failed: too few records to evaluate.")


## Approve Model is Registry

In [ ]:
# model_package_arn = "arn:aws:sagemaker:us-east-1:380537322556:model-package/ReadmissionModelGroup/10"

# response = sagemaker_client.update_model_package(
#     ModelPackageArn=model_package_arn,
#     ModelApprovalStatus="Approved"
# )

# print("Model package approved:", model_package_arn)


## Endpoint Deletion

Uncomment and run following cell to delete endpoint

In [ ]:
# from sagemaker.predictor import Predictor

# predictor = Predictor(endpoint_name=endpoint_name, sagemaker_session=sagemaker_session)
# predictor.delete_endpoint()
# print("Endpoint deleted.")
